In [3]:
from datasets import load_dataset
import torch as t
from transformers import AutoModelForCausalLM, AutoTokenizer
import sys
sys.path.append("..")
from sparsify.sparsify import Sae
from huggingface_hub import hf_hub_download
from safetensors import safe_open
import json
from types import SimpleNamespace # Import SimpleNamespace

# Manually load SAE due to state_dict key mismatch
repo_id = "fnlp/Llama-Scope-R1-Distill"
file_name = "400M-Slimpajama-400M-OpenR1-Math-220k/L15R"
device = "cuda"

# Download necessary files
config_path = hf_hub_download(repo_id=repo_id, filename=f"{file_name}/config.json")
safetensors_path = hf_hub_download(repo_id=repo_id, filename=f"{file_name}/sae_weights.safetensors")

# Load config
with open(config_path, 'r') as f:
    sae_cfg = json.load(f)
    print(sae_cfg)

# Convert dict to SimpleNamespace for attribute access
sae_cfg['device'] = device
sae_cfg['num_latents'] = 32768
sae_cfg['transcode'] = False
sae_cfg['normalize_decoder'] = False
sae_cfg['skip_connection'] = False
sae_cfg["shuffle_seed"] = 42
sae_cfg["k"] = 50
sae_cfg["activation"] = "topk"
# sae_cfg["dead_feature_threshold"] = 10000000
sae_cfg["multi_topk"] = False
sae_cfg["threshold"] = 0.0
sae_cfg_obj = SimpleNamespace(**sae_cfg)
sae = Sae(cfg=sae_cfg_obj, d_in=4096)

{'sae_type': 'sae', 'hook_point_in': 'blocks.15.hook_resid_post', 'hook_point_out': 'blocks.15.hook_resid_post', 'd_model': 4096, 'expansion_factor': 8, 'use_decoder_bias': True, 'use_glu_encoder': False, 'act_fn': 'jumprelu', 'jump_relu_threshold': 0.0, 'apply_decoder_bias_to_pre_encoder': False, 'norm_activation': 'dataset-wise', 'sparsity_include_decoder_norm': False, 'force_unit_decoder_norm': False, 'top_k': 50, 'use_triton_kernel': True, 'sparsity_threshold_for_triton_spmm_kernel': 0.99, 'jumprelu_threshold_window': 2.0}


In [4]:
from autointerp.vis.dashboard import make_feature_display

cache_path = "/share/u/koyena/llama-8b-cache-three/model.layers.15"

features = list(range(100))
feature_display = make_feature_display([cache_path], features)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [ ]:
# check what tokens are in the cache
cache_path = "/share/u/koyena/llama-8b-cache-two-k-128/model.layers.15"
cache = t.load(f"{cache_path}/0.pt")
print(cache)
# see all the tokens in the cache
print(cache['activations'].shape)


{'locations': tensor([[   0,    1,  195],
        [   0,    1,  636],
        [   0,    2,  195],
        ...,
        [ 975, 1023,   72],
        [ 975, 1023,  380],
        [ 975, 1023,  382]]), 'activations': tensor([1.8371, 4.2955, 1.8371,  ..., 0.3498, 0.3685, 0.3694]), 'tokens_path': '/share/u/koyena/llama-8b-cache-two-k-128/tokens.pt', 'model_id': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B'}
torch.Size([2467557])


In [5]:

from autointerp.vis.dashboard import make_feature_display
import os # Add os import if not already there

cache_path = "/share/u/koyena/llama-8b-cache-three/model.layers.15"

features_to_load = list(range(100))
# Assuming the hookpoint name is the last part of the cache_path directory
hookpoint_name = os.path.basename(cache_path)
features_dict = {hookpoint_name: features_to_load}

# Also add the ctx_len argument back, as it was needed before
feature_display = make_feature_display([cache_path], features_dict, ctx_len=100)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [6]:
from autointerp.vis.dashboard import make_dashboard
sae.to("cuda")
cache_path = "/share/u/koyena/llama-8b-cache-three/model.layers.15"
dashboard = make_dashboard(cache_path, sae.simple_encode, in_memory=False,  ctx_len=100)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [4]:
import pandas as pd
df = pd.read_parquet("/share/u/koyena/llama-8b-cache-three/model.layers.15/header.parquet")
print(df.info())
print(f'\nFeature 22219 exists: {22219 in df['feature_idx'].values}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32299 entries, 0 to 32298
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   feature_idx  32299 non-null  int64
 1   shard        32299 non-null  int64
dtypes: int64(2)
memory usage: 504.8 KB
None

Feature 22219 exists: True
